# Conditioning and accuracy across number systems

The condition number κ(A) = σ_max / σ_min is the one number that predicts how
much accuracy a linear solve can keep. For a backward-stable solver, such as LU
with partial pivoting, the rule of thumb is

$$\frac{\lVert \hat x - x \rVert}{\lVert x \rVert} \;\lesssim\; \kappa(A)\, u ,$$

where u is the unit roundoff of the arithmetic. So log₁₀ κ is roughly the number
of decimal digits the solve loses, whatever number system it runs in.

This notebook makes that visible, using a generator that sets κ exactly,
`randsvd(m, n, kappa, mode, seed=...)`, across eight number systems at three
widths:

| Width | IEEE | posit | takum | cascade |
|---|---|---|---|---|
| 16 bits | `fp16` | `posit16` | `takum16` | |
| 32 bits | `f32` | `posit32` | `takum32` | |
| 64 bits and more | `f64` | | | `dd_cascade` (double-double) |

The sections:

1. **Singular values against the unit roundoff:** how a spectrum looks, and what
   rounding to each format does to it.
2. **Requested against achieved conditioning:** rounding a dense matrix to a
   format caps its condition number where rounding noise takes over.
3. **Forward error against κ·u:** the rule of thumb, measured.
4. **The precision ladder:** correct digits for every format and every κ, on one
   chart.

Every random matrix has a seed, so the figures are the same on every build.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

import mtl5

g = mtl5.generators

# Chart styling, shared with the generators notebook: recessive axes and grid,
# ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2, RULE = "#0b0b0b", "#52514e", "#a9a8a3"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

# Colour follows the family, and each width gets its own panel, so no panel has
# more than three series.
FAMILY = {"IEEE": SERIES[0], "posit": SERIES[1], "takum": SERIES[2], "cascade": SERIES[3]}
MARKER = {"IEEE": "o", "posit": "s", "takum": "^", "cascade": "D"}
WIDTHS = {
    "16 bits": [("fp16", "IEEE"), ("posit16", "posit"), ("takum16", "takum")],
    "32 bits": [("f32", "IEEE"), ("posit32", "posit"), ("takum32", "takum")],
    "64 bits and more": [("f64", "IEEE"), ("dd_cascade", "cascade")],
}
FORMATS = [name for group in WIDTHS.values() for name, _ in group]
FAMILY_OF = {name: fam for group in WIDTHS.values() for name, fam in group}

print(mtl5.__version__)

## The unit roundoff of each format

u is half the gap between 1 and the next representable number above it, so
rounding a number near 1 has a relative error of at most u. The cell below
measures it, by finding the largest power of two δ for which 1 + δ rounds back
to 1.

Two caveats:

- **Tapered formats.** posit and takum precision is **tapered**: it is highest
  near 1 and falls off toward the ends of the range. Their u at 1 is the best
  case. The matrices below are scaled so that σ_max = 1, which keeps them in
  the accurate region.
- **The double-double.** `dd_cascade` carries about 106 significand bits, more
  than a float64 can show, so this probe can't measure it from Python. Its u is
  set to 2⁻¹⁰⁶ instead.

In [ ]:
def unit_roundoff(dtype):
    # The largest delta = 2**-k for which 1 + delta rounds to 1.
    for k in range(1, 53):
        if mtl5.convert(np.array([1.0 + 2.0**-k]), dtype).to_numpy()[0] == 1.0:
            return 2.0**-k
    return 2.0**-53


U = {name: unit_roundoff(name) for name in FORMATS}
U["dd_cascade"] = 2.0**-106  # beyond what a float64 probe can see

print(f"{'format':11s} {'u at 1':>9s} {'digits':>7s}")
for name in FORMATS:
    print(f"{name:11s} {U[name]:9.1e} {-np.log10(U[name]):7.1f}")

At 16 bits, posit16 and takum16 each have one more bit than fp16 near 1. That
is the bit their tapering saves on the exponent. At 32 bits, posit32 and takum32
have four more than f32: u = 2⁻²⁸ against 2⁻²⁴. That is about 1.2 extra
decimal digits, as long as the values stay near 1.

## 1. Singular values against the unit roundoff

`randsvd(n, n, kappa, mode)` builds U·Σ·Vᵀ from random orthogonal U and V and
singular values between 1 and 1/κ. The `mode` sets how they are spread:

| mode | singular values |
|---|---|
| 1 | one large: 1, 1/κ, …, 1/κ |
| 2 | one small: 1, …, 1, 1/κ |
| 3 | geometric: σᵢ = κ^(−i/(n−1)) |
| 4 | arithmetic: from 1 down to 1/κ in equal steps |
| 5 | random, log-uniform in [1/κ, 1] |

All five modes have the same κ. They differ in *how many* singular values fall
below a format's unit roundoff, and so in how many directions that format can
resolve at all.

In [ ]:
n, kappa = 32, 1e12
modes = {1: "one large", 2: "one small", 3: "geometric", 4: "arithmetic", 5: "random"}
REF_LINES = [("fp16", "fp16 u"), ("f32", "f32 u"), ("f64", "f64 u")]

fig, axes = plt.subplots(1, 5, figsize=(13, 3.2), sharey=True, constrained_layout=True)
for ax, (mode, label) in zip(axes, modes.items()):
    A = g.randsvd(n, n, kappa, mode, seed=11)
    s = mtl5.svdvals(A)
    ax.semilogy(
        np.arange(1, n + 1),
        s / s[0],
        color=SERIES[0],
        linewidth=2,
        marker="o",
        markersize=3.5,
        zorder=3,
    )
    for fmt, text in REF_LINES:
        ax.axhline(U[fmt], color=RULE, linewidth=1, linestyle="--", zorder=1)
        if mode == 5:
            ax.text(n + 1, U[fmt], text, va="center", ha="left", fontsize=8, color=INK_2)
    ax.set_title(f"mode {mode}: {label}", fontsize=10)
    ax.set_xlabel("index i")
    ax.set_ylim(1e-18, 3)
axes[0].set_ylabel("σᵢ / σ₁")
fig.suptitle(
    f"randsvd({n}, {n}, kappa={kappa:.0e}, mode): the same κ, five spectra", color=INK, fontsize=11
)
plt.show()

Every panel reaches 10⁻¹², but by different routes. Mode 2 has a single small
singular value, which makes the matrix nearly rank-deficient in one direction.
Mode 1 is the opposite: it has a single *large* singular value, so the matrix is
close to rank 1. The dashed lines mark each IEEE width's unit roundoff.
Singular values below a line can't be told apart from rounding noise in that
format.

What does rounding the matrix to a format actually do to its spectrum? Rounding
each entry is a perturbation E with ‖E‖ of order u‖A‖. By Weyl's inequality,
every singular value moves by at most ‖E‖, so nothing below about u·σ₁ can be
trusted. In practice the tail levels off a decade or two lower, at the size of
the rounding noise itself: the format's **numerical rank floor**.

In [ ]:
A = g.randsvd(n, n, kappa, 3, seed=11)
exact = mtl5.svdvals(A)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True, constrained_layout=True)
for ax, (title, group) in zip(axes, WIDTHS.items()):
    ax.semilogy(
        np.arange(1, n + 1),
        exact,
        color=RULE,
        linewidth=1.5,
        linestyle="--",
        label="exact (float64)",
        zorder=2,
    )
    for name, fam in group:
        # The singular values of the ROUNDED matrix, computed in float64: every
        # format here converts to float64 exactly, so this sees fl(A) itself.
        rounded = mtl5.matrix(mtl5.convert(A.to_numpy(), name).to_numpy())
        ax.semilogy(
            np.arange(1, n + 1),
            mtl5.svdvals(rounded),
            color=FAMILY[fam],
            marker=MARKER[fam],
            markersize=3.5,
            linewidth=1.5,
            label=name,
            zorder=3,
        )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("index i")
    ax.set_ylim(1e-18, 3)
    ax.legend(frameon=False, fontsize=8, loc="lower left")
axes[0].set_ylabel("σᵢ of the matrix rounded to the format")
fig.suptitle(
    "Rounding to a format flattens the spectrum at its noise floor (mode 3, κ = 1e12)",
    color=INK,
    fontsize=11,
)
plt.show()

Each format keeps the singular values above its floor and replaces the rest
with noise:

- **16 bits:** the floor is near 10⁻⁵, so only the top ten or so of the 32
  singular values survive.
- **32 bits:** f32 departs from the exact curve near 10⁻⁸. posit32 and takum32
  follow it about a decade further, the extra bits from the table above.
- **f64:** holds all 32 singular values, since 10⁻¹² is well above its 10⁻¹⁶
  floor.
- **`dd_cascade`:** rounding a float64 matrix to double-double is exact, so its
  curve is the exact one.

## 2. Requested against achieved conditioning

The same rounding sets a ceiling on conditioning. Rounding A to a format gives
fl(A) = A + E. Once A's smallest singular value drops below the noise floor of
section 1, the noise decides σ_min instead of A. So κ(fl(A)) follows the
requested κ until it reaches σ₁ over the noise floor, then scatters around that
level. Past that point, the rounded matrix's smallest singular values are
noise, not the ones requested.

This is a property of rounding a *dense* matrix, not a limit on what a format
can store. f32 holds diag(1, 2⁻⁴⁰) exactly, with κ = 2⁴⁰, far above f32's
plateau below. Every entry of a `randsvd` matrix, though, mixes all the
singular values, and rounding those entries buries the small ones.

In [ ]:
n = 24
kappas = np.logspace(0, 16, 33)

achieved = {name: [] for name in FORMATS}
for i, k in enumerate(kappas):
    A = g.randsvd(n, n, float(k), 3, seed=200 + i).to_numpy()
    for name in FORMATS:
        rounded = mtl5.matrix(mtl5.convert(A, name).to_numpy())
        achieved[name].append(mtl5.condition_number(rounded))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True, constrained_layout=True)
for ax, (title, group) in zip(axes, WIDTHS.items()):
    ax.loglog(
        kappas,
        kappas,
        color=RULE,
        linewidth=1.5,
        linestyle="--",
        label="achieved = requested",
        zorder=2,
    )
    for name, fam in group:
        ax.loglog(
            kappas,
            achieved[name],
            color=FAMILY[fam],
            marker=MARKER[fam],
            markersize=4,
            linewidth=1.5,
            label=name,
            zorder=3,
        )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("requested κ (randsvd)")
    ax.legend(frameon=False, fontsize=8, loc="upper left")
axes[0].set_ylabel("κ of the matrix rounded to the format")
fig.suptitle(
    "Rounding a dense matrix caps its condition number at the noise level", color=INK, fontsize=11
)
plt.show()

for name in FORMATS:
    print(f"{name:11s} 1/u = {1 / U[name]:8.1e}   largest κ held = {max(achieved[name]):8.1e}")

The ceilings are about 10⁶ at 16 bits and 10⁹ to 10¹¹ at 32 bits, two to three
decades above 1/u. That is σ₁ over the noise floor of section 1, which sits that
far below u. f64 and `dd_cascade` follow the requested κ all the way, because
the matrices are float64 to begin with, so rounding them changes nothing.

The practical reading: asking a 16-bit format for κ = 10⁸ gives a matrix with κ
of about 10⁶, whose small singular values are rounding noise, not the matrix you
asked for. Before blaming an algorithm, check κ of the matrix the format
actually holds.

## 3. Forward error against κ·u

Now the rule of thumb itself. For each κ, the solve takes three seeded systems
A x = b and runs LU (`mtl5.solve`) in each format.

This is the end-to-end question: given float64 data, how accurate is the
answer when the solve runs in format X? Each test rounds A and b into the
format and solves there. The reference solves the original float64 A and b in
`qd_cascade`, quad-double with about 64 digits. So the measured error includes
rounding the input as well as the LU arithmetic.

Input rounding is itself a backward error of size u, so κ·u still governs it.
The x-axis is the *requested* κ times u. Section 2 showed that the rounded
16- and 32-bit matrices can have a smaller κ than that, once κ·u passes about
1, which is where these errors have already saturated.

If the solve is backward stable, the points fall on or below the diagonal
error = κ·u. The horizontal line at 1 means *no correct digits*.

In [ ]:
def reference(A, b):
    # The same LU solve in quad-double, about 64 digits, on the same float64 data.
    return mtl5.solve(mtl5.convert(A, "qd_cascade"), mtl5.convert(b, "qd_cascade")).to_numpy()


def forward_error(A, b, dtype, ref):
    x = mtl5.solve(mtl5.convert(A, dtype), mtl5.convert(b, dtype)).to_numpy()
    return np.linalg.norm(x - ref) / np.linalg.norm(ref)


n = 24
kappas = np.logspace(0, 16, 17)
seeds = (1, 2, 3)

err = {name: np.empty((len(kappas), len(seeds))) for name in FORMATS}
for i, k in enumerate(kappas):
    for j, seed in enumerate(seeds):
        A = g.randsvd(n, n, float(k), 3, seed=1000 * seed + i).to_numpy()
        x_true = np.random.default_rng(seed).standard_normal(n)
        b = A @ x_true
        ref = reference(A, b)
        for name in FORMATS:
            err[name][i, j] = forward_error(A, b, name, ref)

# Results cross back to Python as float64, so an error below float64's
# resolution reads as 0. Plot those at a floor and say so.
FLOOR = 1e-17

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True, constrained_layout=True)
for ax, (title, group) in zip(axes, WIDTHS.items()):
    for name, fam in group:
        ku = np.repeat(kappas * U[name], len(seeds))
        e = np.maximum(err[name].ravel(), FLOOR)
        ax.loglog(
            ku,
            e,
            linestyle="none",
            marker=MARKER[fam],
            markersize=5,
            color=FAMILY[fam],
            alpha=0.85,
            label=name,
            zorder=3,
        )
    # The x range covers every plotted point, from each panel's own data.
    ku_all = np.concatenate([kappas * U[name] for name, _ in group])
    lim = np.array([ku_all.min() / 3, ku_all.max() * 3])
    ax.loglog(lim, lim, color=RULE, linewidth=1.5, linestyle="--", zorder=2)
    ax.axhline(1.0, color=INK_2, linewidth=1, zorder=2)
    ax.set_xlim(*lim)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("requested κ · u")
    ax.legend(frameon=False, fontsize=8, loc="upper left")
axes[0].set_ylabel("relative forward error")
axes[0].set_ylim(5e-18, 1e3)
axes[2].text(1e-31, 2.5e-17, "float64 boundary: smaller errors read as 0", fontsize=8, color=INK_2)
fig.suptitle(
    "Forward error tracks κ·u in every format (dashed: error = κ·u)", color=INK, fontsize=11
)
plt.show()

Every format follows the same law. At κ near 1 the error sits a little above
κ·u: the rule of thumb leaves out a modest factor that grows with n. As κ grows,
the points follow the diagonal's slope, at or below it, until the error
saturates near 1. What changes from format to format is only *where* on the κ
axis each one runs out:

- **16 bits:** one correct digit at κ = 10³, none from 10⁴.
- **32 bits:** f32 keeps its last digit to κ ≈ 10⁷, takum32 to 10⁸ and
  posit32 to 10⁹.
- **f64:** still has a digit at κ = 10¹⁶.

`dd_cascade` is the instructive exception. Its points sit on the floor because
its answer agrees with the quad-double reference to every bit a float64 can
hold: κ·u_dd is at most 10⁻¹⁶ here. This is the case for extended precision.
It returns a float64 answer correct to float64's full precision, even for
matrices that f64 itself solves to two digits.

### The mode changes the error, not only κ

κ bounds the error; it doesn't determine it. The plot below solves in f32 for
three `randsvd` modes with the same κ, taking the median over three seeds. The
rule of thumb is a worst case. The error a particular solve shows depends on
how the rounding errors project onto the small singular directions, and the
mode decides how many of those directions there are and how small they are.
Here mode 2, with one small singular value, tracks κ·u closely. The geometric
spectrum of mode 3 stays up to a decade below it for κ above 10⁴.

In [ ]:
n = 24
kappas_m = np.logspace(0, 8, 17)
shown = {1: "one large", 2: "one small", 3: "geometric"}

fig, ax = plt.subplots(figsize=(6.4, 3.8), constrained_layout=True)
for slot, (mode, label) in enumerate(shown.items()):
    e_mode = []
    for i, k in enumerate(kappas_m):
        trials = []
        for seed in (1, 2, 3):
            A = g.randsvd(n, n, float(k), mode, seed=5000 * seed + 100 * mode + i).to_numpy()
            x_true = np.random.default_rng(seed).standard_normal(n)
            b = A @ x_true
            trials.append(forward_error(A, b, "f32", reference(A, b)))
        e_mode.append(np.median(trials))
    ax.loglog(
        kappas_m,
        e_mode,
        color=SERIES[slot],
        marker="o",
        markersize=4,
        linewidth=2,
        label=f"mode {mode}: {label}",
        zorder=3,
    )
ax.loglog(
    kappas_m,
    np.minimum(kappas_m * U["f32"], 10),
    color=RULE,
    linewidth=1.5,
    linestyle="--",
    label="κ·u (f32)",
    zorder=2,
)
ax.set_xlabel("κ (randsvd)")
ax.set_ylabel("relative forward error, f32 solve (median)")
ax.set_title("Same κ, different spectra: f32 LU", fontsize=10)
ax.legend(frameon=False, fontsize=8, loc="upper left")
plt.show()

## 4. The precision ladder

The forward errors of section 3 collapse into one chart: the number of correct
decimal digits, −log₁₀ of the error (the median of the three seeds), for every
format and every κ. Read a row to see where a format runs out. Read a column to
see which formats can still solve a matrix that ill-conditioned.

In [ ]:
median = np.array([np.median(err[name], axis=1) for name in FORMATS])
# A median of 0 means the answers agree in every bit float64 can show: at least
# 16 digits, but no more is measurable. Those cells are labelled "16+".
unresolved = median == 0
digits = np.clip(-np.log10(np.maximum(median, FLOOR)), 0, 16)

# The palette's sequential blue ramp, light (no digits) to dark (all of them).
BLUES = ["#cce3fd", "#9dc8f7", "#6aabf1", "#338ced", "#2370cd", "#1757ac", "#0f4a96", "#0e3d7e"]
cmap = ListedColormap(np.array([plt.matplotlib.colors.to_rgb(c) for c in BLUES]))

fig, ax = plt.subplots(figsize=(11, 3.9), constrained_layout=True)
im = ax.imshow(digits, cmap=cmap, vmin=0, vmax=16, aspect="auto")
ax.set_xticks(range(len(kappas)), [f"1e{int(round(np.log10(k)))}" for k in kappas], fontsize=8)
ax.set_yticks(range(len(FORMATS)), FORMATS)
ax.set_xlabel("κ (randsvd, mode 3)")
ax.grid(False)
for r in range(digits.shape[0]):
    for c in range(digits.shape[1]):
        d = digits[r, c]
        label = "16+" if unresolved[r, c] else f"{d:.0f}"
        ax.text(
            c, r, label, ha="center", va="center", fontsize=7.5, color="white" if d >= 8 else INK
        )
cbar = fig.colorbar(im, ax=ax, shrink=0.9)
cbar.set_label("correct decimal digits")
ax.set_title("Correct digits of an LU solve, by format and condition number", fontsize=11)
plt.show()

Each row loses one digit per decade of κ, starting from the format's own digit
count. The diagonal edge across the chart is the line κ·u = 1. Reading off the
chart:

- **The 16-bit formats** are usable only for well-conditioned systems, and
  the three are within half a digit of each other.
- **posit32 and takum32** keep about one digit more than f32 at most κ, the
  bits their tapering saves near 1, so they run out one to two decades of κ
  later. Matrices whose entries are far from 1 would lose that edge: see the
  planned dynamic-range section of #166.
- **f64** runs out around κ = 10¹⁶.
- **`dd_cascade`** agrees with the quad-double reference in every digit a
  float64 can show, across the whole range. Those cells read "16+": the
  answers cross into Python as float64, so digits beyond 16 can't be measured
  here.

## What to take away

- **log₁₀ κ is the digit budget.** A solve in a format with d digits keeps about
  d − log₁₀ κ of them. That holds in every number system here, IEEE, posit,
  takum and cascade alike.
- **Rounding sets a ceiling on conditioning.** Rounding a dense matrix to a
  format caps its condition number where rounding noise takes over, a few
  decades above 1/u. Beyond that cap, the format holds a different matrix from
  the one you asked for.
- **The spectrum's shape matters, not only κ.** `randsvd`'s modes give the same
  κ with different error behaviour. The generators make that a controlled
  variable.
- **Seeds make it a measurement.** Every matrix here comes from
  `randsvd(..., seed=)`, so these numbers can be rerun, compared and cited.

### Where to go next

- `generators.ipynb`: the rest of the test-matrix catalog.
- `mtl5.mixed`: run the accumulation in a wider format than the storage.
- Coming in #166: spectra and inertia, non-normality and pseudospectra, the
  stability of factorizations, and Krylov convergence against spectrum shape.